# PSA_TreeWriter: PSA のヒットの TTree

PSA のモジュールが求めた量を、**1 ヒット(1 ボードの 1 ch)1 行**の TTree `PSA_TreeWriter/psa` に書きます。
`psa_decode.py` の `.h5` と `pixel_id.py` の `_pixel.h5` の中身を合わせたものに当たります。
`PSA_CenterGravity`、`PSA_Calibration` がチェーンに無いときは、その列は書きません。

| 列 | 型 | 中身 | 単位 | 作るモジュール |
|---|---|---|---|---|
| `sevt_id` | uint | `PSA_ReadData` のイベントの番号(0 から)。同じ値の行が同じトリガー | | PSA_ReadData |
| `trig` | uint | トリガーカウンタ(全ボード共通。`ReadData` と一緒のときは TPC のイベント ID と同じ) | | PSA_ReadData |
| `epoch` | int | トリガーカウンタが戻った回数(PSA だけのとき) | | PSA_ReadData |
| `clk` | uint | ボードのクロック | 1 µs | PSA_ReadData |
| `time` | double | ボードの最初のフレームからの時刻 | s | PSA_ReadData |
| `board` | short | HA のボード番号(`ha<NN>` の NN) | | PSA_ReadData |
| `ch` | short | ch 0〜5(= 5 − ADC の番号) | | PSA_ReadData |
| `hitnum` | short | このボードのこのイベントでヒットのある ch の数 | | PSA_ReadData |
| `hg`, `lg` | double | HG、LG の 4 本の積分の和 | ADC × サンプル | PSA_CalcFADC |
| `wave_sum[8]` | float | 8 本(HG0〜3、LG0〜3)それぞれの積分 | ADC × サンプル | PSA_CalcFADC |
| `ped[8]` | float | 8 本それぞれのペデスタル | ADC | PSA_CalcFADC |
| `peak_id` | short | ピークのサンプル | サンプル | PSA_CalcFADC |
| `satu_h`, `satu_l` | bool | HG、LG の飽和 | | PSA_CalcFADC |
| `temp[6]`, `temp_mean` | float | 温度センサー 6 個とその平均 | °C | PSA_CalcFADC |
| `x_hg`, `y_hg`, `x_lg`, `y_lg` | double | 4 隅の重心 | | PSA_CenterGravity |
| `x_corr`, `y_corr` | double | 補正後の位置(ピクセル n, m の中心が整数)。位置の較正の無い ch は NaN | ピクセル | PSA_Calibration |
| `pix_n`, `pix_m` | short | ピクセル 1〜8。8 × 8 の外は −1 | | PSA_Calibration |
| `pixel` | short | 64 ch + 8 (n − 1) + (m − 1)。外は −1 | | PSA_Calibration |
| `hg_corr` | double | 温度で直した hg = hg / gain(T)(温度ゲインの表が無いボードは hg と同じ) | ADC × サンプル | PSA_Calibration |
| `energy` | double | エネルギー = p0 + p1 × hg_corr。較正の無いときは −1 | keV | PSA_Calibration |
| `adc[400]` | ushort | 生の波形 `adc[サンプル × 8 + 本]`(`save_waveform: true` のときだけ) | ADC | PSA_ReadData |

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有、git には入れない)
CACHE.mkdir(exist_ok=True)

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    a = np.frombuffer(h.GetArray(), dtype=np.float64, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-11-03 の UVSOR での **HA_cal のラン `20261103_run01`**(PSA だけ、14 ボード、約 840 MB)を使います。
位置の較正(`calib/psa/position_haNN.csv`)を作ったときと同じ日の較正データです。
データは解析用の計算機の D ドライブ(`/mnt/d/UVSOR2026_late`)にあります。
ほかの場所にあるときは、環境変数 `SMILE3_PSA_CAL_DATA` に `UVSOR2026_late` に当たるディレクトリを設定してください。

1 つのランのボードは、2 つの storage に分かれて書かれています。

| storage | ディレクトリ | PSA のボード |
|---|---|---|
| storage_g1 | `killua/<ラン>` | ha00〜ha05 |
| storage_g0 | `calltt/<ラン>` | ha06〜ha16 |

`run_chain.py` には、片方を `--data-dir`、もう片方を `PSA_ReadData.extra_data_dirs` で渡します。

In [ ]:
DATA = Path(os.environ.get("SMILE3_PSA_CAL_DATA", "/mnt/d/UVSOR2026_late"))
RUN = "20261103_run01"
RUN_DIRS = [DATA / "killua" / RUN, DATA / "calltt" / RUN]
for d in RUN_DIRS:
    print("OK     " if d.is_dir() else "MISSING", d)

## 2. チェーンを流す

設定ファイル `smile3_psa`(`PSA_ReadData → PSA_CalcFADC → PSA_CenterGravity → PSA_Calibration → PSA_TreeWriter`)で、ラン全体を流します(40 秒ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_psa -o psa_20261103_run01.root \
    --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 \
    --set 'PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]'
```

出力は `tutorials/cache/psa_cal_<ラン>.root` に置き、既にあれば流し直しません(PSA のチュートリアルと `4_MakeCalibration/PSA_Energy` で共有)。
`calib/psa` の較正ファイルを変えたときは、このファイルを消して流し直してください。

In [ ]:
def run_chain(out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)"""
    cmd = [sys.executable, str(RUN_CHAIN), "smile3_psa", "-o", str(out), "--data-dir", str(RUN_DIRS[0]),
           "--set", "PSA_ReadData.extra_data_dirs=" + json.dumps([str(RUN_DIRS[1])]), *args]
    print(" ".join(cmd))
    if not out.exists():
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
        r.check_returncode()

OUT = CACHE / f"psa_cal_{RUN}.root"   # 今の calib/psa(位置・温度ゲイン・エネルギー)で流したもの
run_chain(OUT)

## 3. ツリーの中身

ROOT ファイルに入っている列と型を表示します。

In [ ]:
f = ROOT.TFile(str(OUT))
tree = f.Get("PSA_TreeWriter/psa")
print("entries:", tree.GetEntries())
for br in tree.GetListOfBranches():
    print(f"  {br.GetName():10s} {br.GetTitle()}")

## 4. 読み方

`RDataFrame` で読みます。`Filter` で選んでから `AsNumpy` で numpy の配列にするのが簡単です。
`short` の列は numpy の object の配列になるので `astype(int)` で直し、配列の列(`wave_sum` など)は `np.stack` で 2 次元にします。

In [ ]:
df = ROOT.RDataFrame("PSA_TreeWriter/psa", str(OUT))
d = df.Filter("board == 3 && pixel >= 0 && !satu_h").AsNumpy(["sevt_id", "ch", "pix_n", "pix_m", "hg", "wave_sum"])
ch = d["ch"].astype(int)
wave_sum = np.stack([np.asarray(v) for v in d["wave_sum"]])
print(len(ch), "hits of ha03 in 8x8 without saturation; wave_sum:", wave_sum.shape)
print("hits per ch:", np.bincount(ch))

1 イベントのヒットをまとめるときは `sevt_id` で並べます。下は、2 つ以上のボードにヒットがあったイベントの例です。

In [ ]:
a = df.AsNumpy(["sevt_id", "trig", "board", "ch", "hg", "pixel"])
sid = a["sevt_id"]
u, cnt = np.unique(sid, return_counts=True)
multi = u[cnt >= 2][:5]
for e in multi:
    m = sid == e
    print(f"sevt_id {e} (trig {a['trig'][m][0]}):", [(f"ha{int(b):02d}", int(c), round(float(h)), int(p)) for b, c, h, p in zip(a['board'][m], a['ch'][m], a['hg'][m], a['pixel'][m])])

## 5. 波形の列 `adc`

`PSA_TreeWriter.save_waveform: true` で、生の波形 `adc[400]` も書きます(1 ヒット 800 バイト増えます)。
読み方は [3_PSA_CalcFADC](../3_PSA_CalcFADC/PSA_CalcFADC.ipynb) の 3 節を見てください。

## 6. パラメータ

In [ ]:
show_params("smile3_psa", "PSA_TreeWriter")